# Promoter Functional Classification — CNN + BiLSTM + GENERator
**Label rule:** A promoter is *functional* if `max(|expression values|) > 0.5` across all 6 conditions in Supplementary Table 2.

In [ ]:
# ── 1. Imports ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# ── 2. Load & Merge Data ─────────────────────────────────────────────────────
def clean_cols(df):
    df.columns = (
        df.columns.astype(str).str.strip().str.lower()
        .str.replace(r'[\s/]+', '_', regex=True)
        .str.replace(r'[()\n]', '', regex=True)
    )
    return df

t1 = clean_cols(pd.read_excel('capstone_dataset.xlsx', sheet_name='Supplementary Table 1', header=3))
t2 = clean_cols(pd.read_excel('capstone_dataset.xlsx', sheet_name='Supplementary Table 2', header=3))

# Table 1 — keep sequence & metadata
seq_col = next(c for c in t1.columns if 'sequence' in c)
t1 = t1[['gene', 'species', 'gc', 'strand', seq_col]].dropna().rename(columns={seq_col: 'sequence'})

# Table 2 — rename promoter → gene, drop non-numeric cols
t2 = t2.rename(columns={'promoter': 'gene'}).dropna(subset=['gene'])
expr_cols = [c for c in t2.columns if c not in ('gene', 'species')]
t2[expr_cols] = t2[expr_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

# ── NEW LABEL: max(|expr|) > 0.5 → functional ────────────────────────────
t2['max_abs_expr'] = t2[expr_cols].abs().max(axis=1)
t2['label'] = (t2['max_abs_expr'] > 0.5).astype(int)

print('Class distribution in Table 2:')
print(t2['label'].value_counts())

# Merge
data = pd.merge(t1, t2[['gene', 'label'] + expr_cols], on='gene', how='inner').reset_index(drop=True)
print('\nMerged shape:', data.shape)
data.head()

In [ ]:
# ── 3. Feature Engineering ───────────────────────────────────────────────────
MAX_LEN = 200
BASE_MAP = {'A': [1,0,0,0], 'C': [0,1,0,0], 'G': [0,0,1,0], 'T': [0,0,0,1]}

def one_hot_encode(seq, max_len=MAX_LEN):
    seq = str(seq).upper()[:max_len]
    arr = np.zeros((max_len, 4), dtype=np.float32)
    for i, b in enumerate(seq):
        if b in BASE_MAP:
            arr[i] = BASE_MAP[b]
    return arr

# One-hot sequences
X_seq = np.array([one_hot_encode(s) for s in data['sequence']], dtype=np.float32)
print('X_seq:', X_seq.shape)

# Scaled expression features (used only as auxiliary input, NOT for labeling)
scaler = StandardScaler()
X_expr = scaler.fit_transform(data[expr_cols].values).astype(np.float32)
print('X_expr:', X_expr.shape)

# Labels
y = data['label'].values
print('Functional:', y.sum(), '| Non-functional:', (1 - y).sum())

# Metadata
genes_all   = data['gene'].values
species_all = data['species'].values

In [ ]:
# ── 4. Train / Val / Test Split ──────────────────────────────────────────────
idx = np.arange(len(y))
idx_train, idx_temp, y_train, y_temp = train_test_split(
    idx, y, test_size=0.30, stratify=y, random_state=42)
idx_val, idx_test, y_val, y_test = train_test_split(
    idx_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

X_seq_train,  X_seq_val,  X_seq_test  = X_seq[idx_train],  X_seq[idx_val],  X_seq[idx_test]
X_expr_train, X_expr_val, X_expr_test = X_expr[idx_train], X_expr[idx_val], X_expr[idx_test]

print('Train:', X_seq_train.shape)
print('Val  :', X_seq_val.shape)
print('Test :', X_seq_test.shape)

In [ ]:
# ── 5. GENERator Embeddings (run once, then cached) ──────────────────────────
import os

GEN_CACHE = 'X_gen.npy'

if os.path.exists(GEN_CACHE):
    X_gen = np.load(GEN_CACHE)
    print('Loaded cached GENERator embeddings:', X_gen.shape)
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM

    tokenizer = AutoTokenizer.from_pretrained(
        'GenerTeam/GENERator-v2-eukaryote-1.2b-base', trust_remote_code=True)
    gen_model = AutoModelForCausalLM.from_pretrained(
        'GenerTeam/GENERator-v2-eukaryote-1.2b-base').to(device).eval()

    def truncate_to_6(seq):
        r = len(seq) % 6
        return seq[r:] if r else seq

    @torch.no_grad()
    def generator_embeddings(seqs, batch_size=64):
        tokenizer.padding_side = 'left'
        out_all = []
        for i in range(0, len(seqs), batch_size):
            batch = [tokenizer.bos_token + truncate_to_6(s.upper())
                     for s in seqs[i:i+batch_size]]
            inputs = tokenizer(batch, padding=True, return_tensors='pt').to(device)
            outputs = gen_model(**inputs, output_hidden_states=True)
            h = outputs.hidden_states[-1]
            mask = inputs['attention_mask'].unsqueeze(-1).float()
            emb = (h * mask).sum(1) / mask.sum(1)
            out_all.append(emb.cpu())
            if (i // batch_size) % 10 == 0:
                print(f'  Embedded {min(i+batch_size, len(seqs))}/{len(seqs)}')
        return torch.cat(out_all).numpy()

    X_gen = generator_embeddings(data['sequence'].values)
    np.save(GEN_CACHE, X_gen)
    print('Saved GENERator embeddings:', X_gen.shape)

In [ ]:
# ── 6. Dataset & DataLoaders ─────────────────────────────────────────────────
class PromoterDataset(Dataset):
    def __init__(self, seqs, gens, exprs, labels=None):
        self.seqs   = torch.from_numpy(seqs)
        self.gens   = torch.from_numpy(gens.astype(np.float32))
        self.exprs  = torch.from_numpy(exprs)
        self.labels = None if labels is None else torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

    def __len__(self): return len(self.seqs)

    def __getitem__(self, i):
        if self.labels is not None:
            return self.seqs[i], self.gens[i], self.exprs[i], self.labels[i]
        return self.seqs[i], self.gens[i], self.exprs[i]

train_ds = PromoterDataset(X_seq_train, X_gen[idx_train], X_expr_train, y_train)
val_ds   = PromoterDataset(X_seq_val,   X_gen[idx_val],   X_expr_val,   y_val)
test_ds  = PromoterDataset(X_seq_test,  X_gen[idx_test],  X_expr_test,  y_test)

BATCH = 128
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print('DataLoaders ready')

In [ ]:
# ── 7. Model ─────────────────────────────────────────────────────────────────
class PromoterHybridModel(nn.Module):
    def __init__(self, expr_dim, gen_dim, dropout=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(4, 64, kernel_size=8, padding=0),
            nn.ReLU(),
            nn.MaxPool1d(4)
        )
        self.bilstm = nn.LSTM(
            input_size=64, hidden_size=64,
            batch_first=True, bidirectional=True
        )
        self.seq_fc  = nn.Linear(128, 64)
        self.gen_fc  = nn.Linear(gen_dim, 64)
        self.expr_fc = nn.Linear(expr_dim, 32)
        self.drop    = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(160, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1),   nn.Sigmoid()
        )

    def forward(self, seq, gen, expr):
        x = self.cnn(seq.permute(0, 2, 1)).permute(0, 2, 1)  # (B, T', 64)
        _, (h, _) = self.bilstm(x)
        seq_feat  = self.drop(self.seq_fc(torch.cat([h[-2], h[-1]], dim=1)))
        gen_feat  = self.drop(self.gen_fc(gen))
        expr_feat = self.drop(self.expr_fc(expr))
        return self.classifier(torch.cat([seq_feat, gen_feat, expr_feat], dim=1))

model = PromoterHybridModel(X_expr.shape[1], X_gen.shape[1]).to(device)
print(model)

In [ ]:
# ── 8. Training with Validation Monitoring ───────────────────────────────────
# Handle class imbalance
n_neg = int((y == 0).sum())
n_pos = int((y == 1).sum())
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)
print(f'pos_weight = {pos_weight.item():.3f}  (neg={n_neg}, pos={n_pos})')

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

# Redefine model output to use logits (remove Sigmoid from classifier)
model.classifier[-1] = nn.Identity()  # raw logits for BCEWithLogitsLoss
model = model.to(device)

EPOCHS = 20
best_val_f1, best_state = 0, None
history = {'train_loss': [], 'val_loss': [], 'val_f1': []}

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_p, all_l = 0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for seq_b, gen_b, expr_b, lab_b in loader:
            seq_b, gen_b, expr_b, lab_b = (
                seq_b.to(device), gen_b.to(device),
                expr_b.to(device), lab_b.to(device)
            )
            if train: optimizer.zero_grad()
            logits = model(seq_b, gen_b, expr_b)
            loss   = criterion(logits, lab_b)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            all_p.append(torch.sigmoid(logits).cpu().numpy().ravel())
            all_l.append(lab_b.cpu().numpy().ravel())
    probs  = np.concatenate(all_p)
    labels = np.concatenate(all_l).astype(int)
    preds  = (probs > 0.5).astype(int)
    return total_loss / len(loader), f1_score(labels, preds, zero_division=0)

for ep in range(1, EPOCHS + 1):
    tr_loss, tr_f1 = run_epoch(train_loader, train=True)
    vl_loss, vl_f1 = run_epoch(val_loader,   train=False)
    scheduler.step(vl_loss)
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['val_f1'].append(vl_f1)
    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    print(f'Epoch {ep:02d} | Train Loss {tr_loss:.4f}  F1 {tr_f1:.4f} | '
          f'Val Loss {vl_loss:.4f}  F1 {vl_f1:.4f}{" ✓" if vl_f1 == best_val_f1 else ""}')

model.load_state_dict(best_state)
print(f'\nBest Val F1: {best_val_f1:.4f}')

In [ ]:
# ── 9. Training Curves ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'],   label='Val')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(history['val_f1'], label='Val F1', color='green')
axes[1].set_title('Val F1'); axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 10. Optimal Threshold (from validation set) ──────────────────────────────
model.eval()
all_p, all_l = [], []
with torch.no_grad():
    for seq_b, gen_b, expr_b, lab_b in val_loader:
        logits = model(seq_b.to(device), gen_b.to(device), expr_b.to(device))
        all_p.append(torch.sigmoid(logits).cpu().numpy().ravel())
        all_l.append(lab_b.numpy().ravel())

val_probs  = np.concatenate(all_p)
val_labels = np.concatenate(all_l).astype(int)

thresholds = np.arange(0.1, 0.9, 0.01)
best_thresh, best_f1 = 0.5, 0
for t in thresholds:
    f = f1_score(val_labels, (val_probs > t).astype(int), zero_division=0)
    if f > best_f1:
        best_f1, best_thresh = f, t
print(f'Optimal threshold: {best_thresh:.2f}  (Val F1={best_f1:.4f})')

In [ ]:
# ── 11. Test Set Evaluation ───────────────────────────────────────────────────
all_p, all_l = [], []
with torch.no_grad():
    for seq_b, gen_b, expr_b, lab_b in test_loader:
        logits = model(seq_b.to(device), gen_b.to(device), expr_b.to(device))
        all_p.append(torch.sigmoid(logits).cpu().numpy().ravel())
        all_l.append(lab_b.numpy().ravel())

probs  = np.concatenate(all_p)
y_true = np.concatenate(all_l).astype(int)
preds  = (probs > best_thresh).astype(int)

print(f'=== Test Results (threshold={best_thresh:.2f}) ===')
print(f'Accuracy : {accuracy_score(y_true, preds):.4f}')
print(f'Precision: {precision_score(y_true, preds, zero_division=0):.4f}')
print(f'Recall   : {recall_score(y_true, preds, zero_division=0):.4f}')
print(f'F1       : {f1_score(y_true, preds, zero_division=0):.4f}')
print(f'ROC AUC  : {roc_auc_score(y_true, probs):.4f}')

In [ ]:
# ── 12. Per-Species Confusion Matrix & ROC ───────────────────────────────────
output_df = pd.DataFrame({
    'Promoter'        : genes_all[idx_test],
    'Species'         : species_all[idx_test],
    'True_Label'      : ['Functional' if v == 1 else 'Non-functional' for v in y_true],
    'Predicted_Label' : ['Functional' if v == 1 else 'Non-functional' for v in preds],
    'Probability'     : probs
})

species_list = output_df['Species'].unique()
fig, axes = plt.subplots(len(species_list), 2, figsize=(12, 4 * len(species_list)))
if len(species_list) == 1: axes = axes[np.newaxis, :]

for row, sp in enumerate(species_list):
    sub = output_df[output_df['Species'] == sp]
    y_t = sub['True_Label'].map({'Non-functional': 0, 'Functional': 1})
    y_p = sub['Predicted_Label'].map({'Non-functional': 0, 'Functional': 1})
    y_prob = sub['Probability']

    # Confusion Matrix
    cm = confusion_matrix(y_t, y_p)
    ConfusionMatrixDisplay(cm, display_labels=['Non-func', 'Functional']).plot(
        ax=axes[row, 0], cmap='Blues', colorbar=False)
    axes[row, 0].set_title(f'{sp} — Confusion Matrix')

    # ROC
    fpr, tpr, _ = roc_curve(y_t, y_prob)
    auc = roc_auc_score(y_t, y_prob)
    axes[row, 1].plot(fpr, tpr, label=f'AUC={auc:.3f}')
    axes[row, 1].plot([0,1],[0,1],'--', color='grey')
    axes[row, 1].set_xlabel('FPR'); axes[row, 1].set_ylabel('TPR')
    axes[row, 1].set_title(f'{sp} — ROC Curve'); axes[row, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── 13. Full-Dataset Predictions ─────────────────────────────────────────────
full_ds = PromoterDataset(X_seq, X_gen, X_expr)  # no labels
full_loader = DataLoader(full_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

model.eval()
all_full = []
with torch.no_grad():
    for seq_b, gen_b, expr_b in full_loader:
        logits = model(seq_b.to(device), gen_b.to(device), expr_b.to(device))
        all_full.append(torch.sigmoid(logits).cpu().numpy().ravel())

all_probs = np.concatenate(all_full)

# Distribution plot
plt.figure(figsize=(8, 4))
plt.hist(all_probs, bins=60, edgecolor='black')
plt.axvline(best_thresh, color='red', linestyle='--', label=f'Threshold={best_thresh:.2f}')
plt.xlabel('Predicted Probability (Functional)')
plt.ylabel('Count')
plt.title('Predicted Probability Distribution — Full Dataset')
plt.legend()
plt.show()

print('Predicted Non-functional:', (all_probs < best_thresh).sum())
print('Predicted Functional    :', (all_probs >= best_thresh).sum())

In [ ]:
# ── 14. Save Full Results ─────────────────────────────────────────────────────
result_df = pd.DataFrame({
    'Gene'       : genes_all,
    'Species'    : species_all,
    'Probability': all_probs,
    'Prediction' : ['Functional' if p >= best_thresh else 'Non-functional' for p in all_probs]
})
result_df.to_csv('promoter_predictions.csv', index=False)
print('Saved to promoter_predictions.csv')
result_df.head(10)